# Ноутбук для проведения вычислительных экспериментов по оценке точности алгоритма ADD (жадное добавление)

In [8]:
import os

import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import tabulate
import contextily as ctx

# Версии библиотек
print('OSMnx:', ox.__version__,
      '\nGeoPandas:', gpd.__version__,
      '\nPandas:', pd.__version__,
      '\nContextily', ctx.__version__,
      )

ox.settings.default_access = ''

OSMnx: 2.0.1 
GeoPandas: 1.0.1 
Pandas: 2.2.2 
Contextily 1.6.2


In [2]:
DATA_PATH = '_data/add_check_data/'

# Основной цикл

In [3]:
settlements = pd.read_csv(os.path.join(DATA_PATH, 'settlements.csv'), index_col=0)
settlements['joined'] = settlements['joined'].str.strip()
cities_list = settlements['joined'].unique().tolist()

print('Количество городов:', len(cities_list))
cities_list

Количество городов: 93


['Обнинск, Калужская область',
 'город Евпатория, Республика Крым',
 'Сызрань, Самарская область',
 'Кызыл, Республика Тыва',
 'Южно-Сахалинск, Сахалинская область',
 'Старый Оскол, Белгородская область',
 'Невинномысск, Ставропольский край',
 'Сергиев Посад, Московская область',
 'Дербент, Республика Дагестан',
 'Новочеркасск, Ростовская область',
 'Ковров, Владимирская область',
 'Нижнекамск, Республика Татарстан',
 'Бердск, Новосибирская область',
 'Королев, Московская область',
 'Майкоп, Республика Адыгея',
 'Балаково, Саратовская область',
 'Волгодонск, Ростовская область',
 'Благовещенск, Амурская область',
 'Черкесск, Карачаево-Черкесская республика',
 'Шахты, Ростовская область',
 'Жуковский, Московская область',
 'Ногинск, Московская область',
 'Железногорск, Курская область',
 'Елец, Липецкая область',
 'Великий Новгород, Новгородская область',
 'Батайск, Ростовская область',
 'Балашиха, Московская область',
 'Альметьевск, Республика Татарстан',
 'Ангарск, Иркутская область',

In [4]:
# cities_list = [
#     'Ачинск',
#     'Канск',
#     'Железногорск',
#     'Братск',
#     'Бердск',
#     'Novokuznetsk'
# ]

In [ ]:
from fire_units.metrics import CoverIndexBuilding
from fire_units.settings import BUILDING_FIELDS, DEFAULT_SPEEDS
from genesis.core import MetricBase
from genesis.lscp import LSCP_ADD, drop_trash_points
from genesis.states import FirstArrivalUnitState, get_atm
from graphs.speeds import kmh_to_mm, set_graph_travel_times


def get_bounds(city_name, city_name_full, layer_name='bounds'):
    '''
    Получение полигона границ
    '''
    path = f'{DATA_PATH}{city_name}.gpkg'
    try:
        gdf = gpd.read_file(path, layer=layer_name)
    except:
        gdf = ox.geocode_to_gdf(city_name_full)
        if gdf is None:
            raise ValueError(f'Данные для города {city_name_full} не получены!')
        gdf.to_file(path, layer=layer_name)

    print(f'Границы города {city_name} получены!')
    return gdf
        
def get_g(city_name, city_name_full, city_bounds,  simplify = False, retain_all = False, buffer=2000):
    '''
    Получение графа улично-дорожной сети
    '''
    path = f'{DATA_PATH}{city_name}.ml'
    if os.path.exists(path):
        G = ox.load_graphml(path)
    else:
        # G = ox.graph_from_place(city_name_full,
        #                         network_type = 'drive_service',
        #                         simplify     = simplify,
        #                         retain_all   = retain_all,
        #                         )
        city_bounds_buffer = ox.projection.project_gdf(city_bounds.copy())
        city_bounds_buffer.geometry = city_bounds_buffer.geometry.buffer(buffer)
        city_bounds_buffer = ox.projection.project_gdf(city_bounds_buffer, to_latlong=True)

        G = ox.graph_from_polygon(
            city_bounds_buffer.union_all(),
            network_type = 'drive_service',
            simplify     = simplify,
            retain_all   = retain_all,
        )

        if G is None:
            raise ValueError(f'ГДС города {city_name} не получен!')
        
        set_graph_travel_times(
            G,
            speeds         = DEFAULT_SPEEDS,
            morph_function = kmh_to_mm)
        
        ox.save_graphml(G, path)
        g_edges = ox.graph_to_gdfs(G, nodes=False)
        g_edges.to_file(f'{DATA_PATH}{city_name}.gpkg', layer='highways')

    print(f'ГДС города {city_name} получен! Времена следования установлены!')
    return G

def get_buildings(city_name, city_bounds, layer_name='buildings'):
    '''
    Загрузка данных о зданиях
    '''
    path = f'{DATA_PATH}{city_name}.gpkg'
    try:
        buildings = gpd.read_file(path, layer=layer_name)
    except:
        buildings = ox.features_from_polygon(city_bounds, tags={'building': True})
        if buildings is None:
            raise ValueError(f'Здания города {city_name} не получены!')
        # Оставляем только данные с геометрией Полигон или Мультиполигон
        buildings = buildings[buildings.geometry.apply(lambda x: x.geom_type).isin(['Polygon', 'MultiPolygon'])]
        # Оставляем только заданные поля (и при этом имеющиеся в списке полученных)
        fields = list(set(buildings.columns) & set(BUILDING_FIELDS))
        buildings = buildings[fields]
        # Удаляем индекс по типу элемента
        buildings = buildings.reset_index().drop('element', axis=1)
        # Устанавливаем индекс 'id'
        buildings = buildings.set_index('id')

        buildings.to_file(path, layer=layer_name)

    print(f'Здания города {city_name} получены! Всего {len(buildings)} зданий.')

    return buildings

def add_buildings_node(G, buildings, drop_node_duplicates = True):
    '''
    Установка ближайшего узла для зданий
    '''
    centroids = ox.projection.project_gdf(buildings).geometry.centroid
    buildings['node'] = ox.distance.nearest_nodes(ox.project_graph(G), 
                                              centroids.x,
                                              centroids.y
                                              )
    # Отбрасываем дублирующиеся здания (по узлам)
    if drop_node_duplicates:
        t =len(buildings)
        buildings = buildings.drop_duplicates(subset='node')  #.set_index('node') #.sort_index()
        print(f'Размер набора зданий без дубликатов в узлах {len(buildings)} (из {t})')

    print('Здания с ближайшими узлами ГДС сопоставлены.')
    return buildings
    
def plot_map(borders, g_nodes, best_nodes, best_nodes2, city_name):
    ## Улучшенная печать карты
    """
    Печать карты с границами города, узлами "можно удалить" и "достаточное размещение"
    """
    # Настройка стиля графика
    plt.style.use('bmh')

    fig, ax = plt.subplots(figsize=(10, 10))
    borders.plot(ax=ax, color='#fcfcd9', edgecolor='black', linewidth=2, alpha=0.7, zorder=1)

    # Отображение узлов
    g_nodes.loc[best_nodes.keys()].plot(ax=ax, facecolors='r', edgecolors='k', markersize=50, label='Можно удалить')
    g_nodes.loc[best_nodes2.keys()].plot(ax=ax, facecolors='g', edgecolors='k', markersize=50, label='Достаточное размещение')


    # Подписи к узлам
    offset = (float(borders.bounds.iloc[0]['maxy']) - float(borders.bounds.iloc[0]['miny'])) / 100
    for node, unit in best_nodes.items():
        g_node_geometry = g_nodes.loc[node].geometry
        ax.text(
            g_node_geometry.x,
            g_node_geometry.y,
            str(unit),
            fontsize=10, 
            color='black',
            ha='center',
            va='bottom',
            position = (g_node_geometry.x, g_node_geometry.y + offset),
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='k', boxstyle='round,pad=0.2'),
            zorder=5
        )

    # Добавить подложку OSM
    ctx.add_basemap(ax, 
                    crs=g_nodes.crs, 
                    source=ctx.providers.OpenStreetMap.Mapnik, 
                    alpha=0.7, 
                    zorder=0)

    # ax.set_axis_off()
    ax.legend(loc='upper left', fontsize=12, frameon=True)
    ax.set_title(city_name, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{DATA_PATH}imgs/{city_name}.png')
    plt.close()
    # plt.show()


def save_results(city_name, best_nodes, best_metric, best_nodes2, best_metric2):
    path = f'{DATA_PATH}results.csv'
    try:
        results = pd.read_csv(path, index_col='Город')
        results.loc[city_name] = {
            'ПСЧ до': len(best_nodes),
            'ИП-10 до': best_metric,
            'ПСЧ после': len(best_nodes2),
            'ИП-10 после': best_metric2
        }
        # print(tabulate.tabulate(results))
    except:
        results = pd.DataFrame(
            {
                'Город': [city_name],
                'ПСЧ до': len(best_nodes),
                'ИП-10 до': best_metric,
                'ПСЧ после': len(best_nodes2),
                'ИП-10 после': best_metric2                
            }
        )
        results.set_index('Город', inplace=True)

    results.to_csv(path, index=True)


class CoverIndexBuilding2(MetricBase):
    '''
    !Только для верификации ADD

    Класс-функция расчета индекса прибытия к зданиям
    '''
    def __init__(self,
                 buildings,
                #  f:callable = np.mean,
                 comp_func:callable = max,
                 zero_val=0,
                 ip_val=10,
                 buildings_node_id_field: str = 'node',
                #  buildings_demand_field: str = 'demand',
                 ) -> None:
        '''
        `buildings`: pd.GeoDataFrame
            Геодатафрейм со сведениями о зданиях.
        `f`: callable
            Функция расчета показателя. По-умолчанию = np.mean,
            т.е. вычисляется среднее время следования.
        `comp_func`: callable
            Функция сравнения значений метрики.
            По умолчанию - лучшей считается большая.
        `zero_val`: float = 0
            Значение которое будет возвращено в случае передачи набора данных `route_times` без элементов.
        `buildings_node_id_field`: str = 'node'
            Имя поля в `buildings` содержащего идентификаторы узлов.
        buildings_demand_field: str = 'demand'
            Имя поля в `buildings` содержащего значения спроса
        '''
        if not buildings_node_id_field in buildings.columns:
            raise KeyError(f"Поле `{buildings_node_id_field}` отсутствует в 'buildings'")

        self.buildings = buildings
        # self.f = f
        self.zero_val = zero_val
        self.buildings_node_id_field = buildings_node_id_field
        # self.buildings_demand_field = buildings_demand_field
        self.ip_val = ip_val
        super().__init__(comp_func)

    def __call__(self, state, area=None):
        '''
        `state` (`состояние`): pd.Series
            Серия времен прибытия в разные точки окружения.

        `area`: pd.Series
            Серия данных содержащих маску точек которые должны быть учтены при расчете метрики.
        '''

        if not isinstance(state, (list, pd.Series)):
            raise TypeError(
                f"Аргумент state может быть только типа list или pd.Series"
                f" Имеется {type(state)}"
                )

        
        # Отбор узлов по area (списку узлов которые следует учесть в расчете)
        if not area is None:
            if isinstance(state, pd.Series):
                state_c = state.loc[area]
            else:
                state_c = [x for x in state if x in area]
        else:
            state_c = state

        if len(state_c)==0:
            return self.zero_val

        # Расчет метрики по спросу
        buildings_s = self.buildings.query(f'{self.buildings_node_id_field} in @state_c.keys()') # Вот здесь!!!
        merged_df = pd.merge(buildings_s,
                             state_c,
                             how='left',
                             left_on=self.buildings_node_id_field,
                             right_index=True)
        # Расчет
        state_c = merged_df['times']
        if len(state_c)==0:
            return self.zero_val
        # ip_len = sum([1 for t in state_c if t<=self.ip_val])
        ip_len = sum(state_c <= self.ip_val)
        # tot_len = len(state_c)
        # только для проверки ADD:
        tot_len = len(self.buildings)

        return 100*ip_len/tot_len


def check_step(city_name_full, aim_ip=95, drop_node_duplicates=True, buildings_size=10000):

    # Уточняем имена населенных пунктов
    city_name      = city_name_full.split(', ')[0]
    city_name_full = city_name_full + ', Россия'

    print(f'Расчет для города "{city_name}"')

    # 1. Загрузка данных
    # Границы города
    borders = get_bounds(city_name, city_name_full)
    city_bounds = borders.union_all()

    # Загрузка ГДС
    G = get_g(city_name, 
              city_name_full, 
              borders,
              retain_all=True)

    # Загрузка зданий
    buildings = get_buildings(city_name, city_bounds)
    buildings = add_buildings_node(G, buildings, drop_node_duplicates = drop_node_duplicates)


    # 2. Расчет ADD
    # Определение выборки по зданиям
    if len(buildings) > buildings_size:
        print(f'Объем выборки: {buildings_size} зданий')
        buildings_set = buildings.sample(buildings_size)
    else:
        print(f'Объем выборки: {len(buildings)} зданий (все)')
        buildings_set = buildings
    # Определение узлов лежащих в пределах полигонов
    g_nodes = ox.graph_to_gdfs(G, edges=False)
    nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)
    # print('1')
    # print(nodes_in_bounds)
    # print('2')
    # Матрица
    matrix = get_atm(G,
                 data       = buildings_set,
                 cutoff     = 10,
                 target_set = nodes_in_bounds
                 )

    # 3. Сборка и инициализация алгоритма
    ## Выбор алгоритма моделирования параметров прибытия
    state = FirstArrivalUnitState()

    ## Выбор метрики
    metric = CoverIndexBuilding2(buildings_set, ip_val=10) # Используется метрика ИП-10. Для ADD метрика выполняет роль оценки

    ## Функция остановки
    stop_case = lambda value, **kwargs: value >= aim_ip

    ## Функция выполняемая при добавлении новой ПСЧ
    after_mclp = lambda best_metric, dynamic_nodes, **kwargs: print('Подразделений:', len(dynamic_nodes), ' ИП-10:', best_metric)

    ## Сборка и инициализация алгоритма ADD
    add = LSCP_ADD(  
        state_function      = state,
        matrix              = matrix,
        metric_function     = metric,
        stop_case_function  = stop_case,
        after_mclp_function = after_mclp,
    )

    # 3. Расчет
    best_nodes, best_metric = add(
            env        = G,
            # nodes_list = nodes_in_bounds,
        )


    # 4. Печать результатов
    print('\t', 'ИТОГ:')
    print('\t', 'Подразделений:', len(best_nodes), ' ИП-10:', best_metric)

    ## Получение набора данных узлов графа, для сопоставления с полученным лучшим узлом
    g_nodes = ox.graph_to_gdfs(G, edges=False)
    # g_nodes.to_file(f'{DATA_PATH}{city_name}.gpkg', layer='add')


    # 5. Отброс мусорных размещений
    best_nodes2, best_metric2 = drop_trash_points(G,
                    state_function      = state,
                    metric_function     = metric,
                    stop_case_function  = stop_case,
                    dynamic_nodes       = best_nodes,
                      )
    print('\t', 'После отброса:')
    print('\t', 'Подразделений:', len(best_nodes2), ' ИП-10:', best_metric2)

    # Печать итога
    df = pd.DataFrame({'Подразделений': [len(best_nodes), len(best_nodes2)],
                       'ИП-10': [best_metric, best_metric2]}, index = ['ADD', 'После отброса'])
    print(tabulate.tabulate(df, headers='keys', tablefmt='grid', showindex=True))

    # 6. Сохранение результатов
    save_results(city_name, best_nodes, best_metric, best_nodes2, best_metric2)

    ## Печать карты
    plot_map(borders, g_nodes, best_nodes, best_nodes2, city_name_full)



# НЕПОСРЕДСТВЕННО ВЫЧИСЛЕНИЯ
# check_step(cities_list[0], aim_ip=100, drop_node_duplicates=True, buildings_size=10000)

Итеративный расчет для всех городов

In [ ]:

for i, city_name in enumerate(cities_list):
    print('\t', f'{i} из {len(cities_list)}')

    try:
        check_step(city_name, aim_ip=100, buildings_size=10000, drop_node_duplicates=True)
    except KeyboardInterrupt:
        print('='*80)
        print('Выполнение кода расчетов остановлено пользователем')
        print(f'Рассчитано {i} из {len(cities_list)} городов')
        break
    except Exception as e:
        print(e)
    finally:
        print('='*80)

In [ ]:
# city_name = 'Сызрань, Самарская область'
# check_step(city_name, aim_ip=100, buildings_size=500, drop_node_duplicates=True)

# Расчет количества покрываемых зданий для узлов ГДС

In [27]:
def save_cover_count(city_name_full, ip = 10, drop_node_duplicates=True, buildings_size=10000):

    # Уточняем имена населенных пунктов
    city_name      = city_name_full.split(', ')[0]
    city_name_full = city_name_full + ', Россия'

    print(f'Расчет для города "{city_name}"')

    # 1. Загрузка данных
    # Границы города
    borders = get_bounds(city_name, city_name_full)
    city_bounds = borders.union_all()

    # Загрузка ГДС
    G = get_g(city_name, 
              city_name_full, 
              borders,
              retain_all=True)

    # Загрузка зданий
    buildings = get_buildings(city_name, city_bounds)
    buildings = add_buildings_node(G, buildings, drop_node_duplicates = drop_node_duplicates)


    # 2. Расчет ADD
    # Определение выборки по зданиям
    if len(buildings) > buildings_size:
        print(f'Объем выборки: {buildings_size} зданий')
        buildings_set = buildings.sample(buildings_size)
    else:
        print(f'Объем выборки: {len(buildings)} зданий (все)')
        buildings_set = buildings
    # Определение узлов лежащих в пределах полигонов
    g_nodes = ox.graph_to_gdfs(G, edges=False)
    nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)

    # Матрица
    matrix = get_atm(G,
                 data       = buildings_set,
                 cutoff     = 10,
                 target_set = nodes_in_bounds
                 )
    
    
    cover_counts = pd.Series(
        np.sum(matrix <= ip, axis = 0),
        name='cover_count',
    )

    g_nodes2 = pd.merge(g_nodes.copy(), cover_counts, how='left', left_index=True, right_index=True)
    g_nodes2['cover_count'] = g_nodes2['cover_count'].fillna(0)

    path = f'{DATA_PATH}{city_name}.gpkg'
    g_nodes2.to_file(path, layer='cover_count')

    return g_nodes2

save_cover_count(cities_list[0], drop_node_duplicates=False, buildings_size=10000)

Расчет для города "Обнинск"
Границы города Обнинск получены!
ГДС города Обнинск получен! Времена следования установлены!
Здания города Обнинск получены! Всего 5330 зданий.
Здания с ближайшими узлами ГДС сопоставлены.
Объем выборки: 5330 зданий (все)
|########################################| 100.0%


# Тесты

## Проверка корректности расчета матрицы

1. Есть подозрение, что в некоторых случаях в матрице могут появиться здания, к которым нет времени прибытия из узлов.
2. Уточнить где устанавливается `delay` для матрицы. 
3. НЕОБХОДИМО проверить корректность расчета ИП между матричным подходом и FirstUnitState


1. Ошибка возникает, когда происходит ограничение возможных мест размещения ПСЧ. Устранено - добавлена проверка отброса зданий непосредственно в процедуре `get_atm()`.
2. Перед расчетом он убирается, после - прибавляется. Таки образом, cutoff рассчитывается как `10-delay`. Но в итоге получается как `10`.
3. Добавил специальную метрику, которая учитывает ровно те же здания, что и были переданы (без сопоставления с достижимыми узлами). Применяется только для проверки ADD.

### Проверка 1,2

In [12]:
import networkx as nx
import numpy as np

from genesis.swiss_knife import DELAY_TIME
from genesis.utils import Progressbar


def get_atm(G,
            data: gpd.GeoDataFrame = None,
            data_sample_size: int  = None,
            data_node_field: str   = 'node',
            weight: str            = 'travel_time',
            cutoff: float          = None,
            data_cutoff_field: str = None,
            delay: float           = DELAY_TIME,
            target_set: set        = None,
            ):
    '''
    Расчет матрицы времен прибытия.

    Аргументы:

        `G`: nx.MultiDiGraph
            Граф улично-дорожной сети

        `data`: pd.DataFrame = None
            Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
            например, здания, или набор узлов графа.
            По умолчанию используются все узлы графа

        `data_sample_size`: int = None
            Размер выборки данных для расчета. Количество записей
            data ,которые будут учтены при расчете.
        
        `data_node_field`: str   = 'node'
            Поле в котором хранится индекс узла, для которого производится расчет.

        `weight`: str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.

        `cutoff`: float = None,
            Размер расчетной области. По умолчанию производится расчет для всего графа.

        `delay`: float, optional = None
            Задержка в расчете. Например на обслуживание вызова на пожар.
            По умолчанию указана в swiss_knife.DELAY_TIME

        `target_set`: set = None
            Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

    Возвращает:

        `matrix`: pd.DataFrame
            Время прибытия первого подразделения в каждый из узлов графа. 
            Столбцами являются узлы графа, индексами - узлы графа, до которых производился расчет.
    '''

    # Если конкретный набор данных не передан, используем узлы графа
    if data is None:
        data = ox.graph_to_gdfs(G, edges=False)
        data[data_node_field] = data.index

    # Если указана доля набора данных, выбираем его случайным образом из data
    if not data_sample_size is None:
        data = data.sample(data_sample_size)

    # Реверсный граф
    GR = nx.reverse(G, copy=True)

    # перебираем все записи в наборе данных
    d = {}
    pb = Progressbar(len(data), bins=40)
    for di, dt in data.iterrows():
        node = dt[data_node_field]
        if not data_cutoff_field is None:
            ctf = dt[data_cutoff_field]
        else:
            ctf = cutoff - delay if not cutoff is None else None
        length = nx.single_source_dijkstra_path_length(
            GR,
            source = node,
            cutoff = ctf,
            weight = weight,
            )
        # if not target_set is None:
        #     length = {k:v for k,v in length.items() if k in target_set}
            
        d[di] = pd.Series(length) + delay
        # d[node] = pd.Series(length) + delay
        pb()

    # Формирование матрицы
    matrix = pd.DataFrame(d).T
    print(matrix.shape)

    # Удаление зданий, к которым невозможно прибытия из перечня приемлемых узлов
    if not target_set is None:
        # Оставляем только узлы в которых можно разместить
        matrix = matrix[list(set(matrix.columns) & target_set)]
        # Здания к которым невозможно прибыть после отброса
        zero_buildings = np.sum(matrix, axis=1)==0
        zero_buildings_count = sum(zero_buildings)
        if zero_buildings_count > 0:
            print(f'!{zero_buildings_count} зданий не доступны из указанных мест размещения!')
            matrix = matrix[zero_buildings == False]

    del GR
    return matrix



#===============================================================================
city_name_full = cities_list[2]
print(city_name_full)

# Уточняем имена населенных пунктов
city_name      = city_name_full.split(', ')[0]
city_name_full = city_name_full + ', Россия'

print(f'Расчет для города "{city_name}"')

# 1. Загрузка данных
# Границы города
borders = get_bounds(city_name, city_name_full)
city_bounds = borders.union_all()

# Загрузка ГДС
G = get_g(city_name, 
            city_name_full, 
            borders,
            retain_all=True)

# Загрузка зданий
buildings = get_buildings(city_name, city_bounds)
buildings = add_buildings_node(G, buildings, drop_node_duplicates = True)

# 2. Расчет ADD
# Определение выборки по зданиям
buildings_size = 20000
if len(buildings) > buildings_size:
    print(f'Объем выборки: {buildings_size} зданий')
    buildings_set = buildings.sample(buildings_size)
else:
    print(f'Объем выборки: {len(buildings)} зданий (все)')
    buildings_set = buildings
# Определение узлов лежащих в пределах полигонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)

# Определение узлов лежащих в пределах полигонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)

# Матрица
matrix = get_atm(G,
                data       = buildings_set,
                cutoff     = 10,
                target_set = nodes_in_bounds,
                )






Сызрань, Самарская область
Расчет для города "Сызрань"
Границы города Сызрань получены!
ГДС города Сызрань получен! Времена следования установлены!
Здания города Сызрань получены! Всего 27493 зданий.
Размер набора зданий без дубликатов в узлах 8212 (из 27493)
Здания с ближайшими узлами ГДС сопоставлены.
Объем выборки: 8212 зданий (все)
|########################################| 100.0%
(8212, 25833)


In [13]:
# 3. Сборка и инициализация алгоритма
## Выбор алгоритма моделирования параметров прибытия
state = FirstArrivalUnitState()

## Выбор метрики
metric = CoverIndexBuilding(buildings_set, ip_val=10) # Используется метрика ИП-10. Для ADD метрика выполняет роль оценки

## Функция остановки
stop_case = lambda value, **kwargs: value >= 100

## Функция выполняемая при добавлении новой ПСЧ
after_mclp = lambda best_metric, dynamic_nodes, **kwargs: print('Подразделений:', len(dynamic_nodes), ' ИП-10:', best_metric)

## Сборка и инициализация алгоритма ADD
add = LSCP_ADD(  
    state_function      = state,
    matrix              = matrix,
    metric_function     = metric,
    stop_case_function  = stop_case,
    after_mclp_function = after_mclp,
)

# 3. Расчет
best_nodes, best_metric = add(
        env        = G,
        # nodes_list = nodes_in_bounds,
    )


# 4. Печать результатов
print('\t', 'ИТОГ:')
print('\t', 'Подразделений:', len(best_nodes), ' ИП-10:', best_metric)

## Получение набора данных узлов графа, для сопоставления с полученным лучшим узлом
g_nodes = ox.graph_to_gdfs(G, edges=False)
g_nodes.to_file(f'{DATA_PATH}{city_name}.gpkg', layer='add')


# 5. Отброс мусорных размещений
best_nodes2, best_metric2 = drop_trash_points(G,
                state_function      = state,
                metric_function     = metric,
                stop_case_function  = stop_case,
                dynamic_nodes       = best_nodes,
                    )
print('\t', 'После отброса:')
print('\t', 'Подразделений:', len(best_nodes2), ' ИП-10:', best_metric2)

# Печать итога
df = pd.DataFrame({'Подразделений': [len(best_nodes), len(best_nodes2)],
                    'ИП-10': [best_metric, best_metric2]}, index = ['ADD', 'После отброса'])
print(tabulate.tabulate(df, headers='keys', tablefmt='grid', showindex=True))

Подразделений: 1  ИП-10: 53.84141077814774
Подразделений: 2  ИП-10: 70.51424343322235
Подразделений: 3  ИП-10: 78.94931557528672
Подразделений: 4  ИП-10: 85.6455789863115
Подразделений: 5  ИП-10: 91.34295227524973
Подразделений: 6  ИП-10: 94.15464298927118
Подразделений: 7  ИП-10: 95.93044765075841
Подразделений: 8  ИП-10: 97.01566161055617
Подразделений: 9  ИП-10: 97.94055987174744
Подразделений: 10  ИП-10: 98.81613022567518
Подразделений: 11  ИП-10: 99.16152897657213
Подразделений: 12  ИП-10: 99.4327990135635
Подразделений: 13  ИП-10: 99.67940813810111
Подразделений: 14  ИП-10: 99.8150431565968
Подразделений: 15  ИП-10: 99.81524818327381
Подразделений: 16  ИП-10: 99.88914890996428
Подразделений: 17  ИП-10: 99.95073284887302
Подразделений: 18  ИП-10: 99.95075710944232
Подразделений: 19  ИП-10: 99.95077528919518
Подразделений: 20  ИП-10: 99.95079345552959
Подразделений: 21  ИП-10: 99.9508116084604
Подразделений: 22  ИП-10: 99.95082974800246
Подразделений: 23  ИП-10: 99.97541487400123
П

### Проверка 3

In [63]:
city_name_full = cities_list[0]
print(city_name_full)

# Уточняем имена населенных пунктов
city_name      = city_name_full.split(', ')[0]
city_name_full = city_name_full + ', Россия'

print(f'Расчет для города "{city_name}"')

# 1. Загрузка данных
# Границы города
borders = get_bounds(city_name, city_name_full)
city_bounds = borders.union_all()

# Загрузка ГДС
G = get_g(city_name, 
            city_name_full, 
            borders,
            retain_all=True)

# Загрузка зданий
buildings = get_buildings(city_name, city_bounds)
buildings = add_buildings_node(G, buildings, drop_node_duplicates = True)

# 2. Расчет ADD
# Определение выборки по зданиям
buildings_size = 50000
if len(buildings) > buildings_size:
    print(f'Объем выборки: {buildings_size} зданий')
    buildings_set = buildings.sample(buildings_size)
else:
    print(f'Объем выборки: {len(buildings)} зданий (все)')
    buildings_set = buildings
# Определение узлов лежащих в пределах полигонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)

# Определение узлов лежащих в пределах полигонов
g_nodes = ox.graph_to_gdfs(G, edges=False)
nodes_in_bounds = set(g_nodes[g_nodes.within(city_bounds)].index)

# Матрица
matrix = get_atm(G,
                data       = buildings_set,
                cutoff     = 10,
                # target_set = nodes_in_bounds,
                )

Обнинск, Калужская область
Расчет для города "Обнинск"
Границы города Обнинск получены!
ГДС города Обнинск получен! Времена следования установлены!
Здания города Обнинск получены! Всего 5330 зданий.
Размер набора зданий без дубликатов в узлах 2851 (из 5330)
Здания с ближайшими узлами ГДС сопоставлены.
Объем выборки: 2851 зданий (все)
|########################################| 100.0%


In [71]:
import warnings
import networkx as nx
import numpy as np
from genesis._errors import KeyNotInMatrixError
from genesis.core import LSCPBase, MCLPBase, MetricBase, PointSelectorBase, StateBase, StopCaseBase
from genesis.mclp import NodesMetric
from genesis.tools import get_duplicates_list, list_dict_concat


class LSCP_ADD(LSCPBase):
    '''
    Решение задачи LSCP алгоритмом жадного добавления.
    '''
    def __init__(self,
                 state_function: StateBase,
                 matrix: pd.DataFrame,
                 metric_function: MetricBase,
                 ip_val: int = 10,
                 mclp_function: MCLPBase = None,
                 point_selector: PointSelectorBase = None,
                 stop_case_function:  StopCaseBase = None,
                 names_pattern:       str = '{}',
                 start_names_index:   int = 1,
                 target_weights:      pd.Series = None,
                 after_mclp_function: callable = None,
                 **kwargs):
        '''
        
        # Аргументы

        `state_function`: StateBase

            Функция расчета состояния узла.

        `matrix`: pd.DataFrame

            Матрица времен прибытия, отражающая время прибытия к каждому из объектов из каждого из узлов.

        `metric_function`: MetricBase

            Функция расчета метрики оптимальности размещения подразделений.
        
        `ip_val`:int

            Пороговое значение для определения индекса прикрытия.
            Рекомендуется использовать 10 для городских населенных пунктов и 
            20 для сельских.

        `mclp_function`: MCLPBase = None,

            Реализация алгоритма решения задачи MCLP.

        `point_selector`: PointSelectorBase = None

            Реализация алгоритма выбора узлов.

        `stop_case_function``: callable = None,

            Функция условия остановки. Если не указана, расчет производится 
            до тех пор пока не будут удалены все строки в matrix.

        `names_pattern`: str = '{}'

            Шаблон имен подразделений.
        
        `start_names_index`: int = 1

            Стартовый номер подразделений.
        
        `after_mclp_function`: callable = None

            Функция выполняемая после выполнения каждой итерации алгоритма.
            
            Сигнатура функции:

                after_mclp_function(iteration,
                            best_metric,
                            current_metric,
                            dynamic_nodes,
                            static_nodes)
        
        '''

        # Сохраняем матрицу прибытия в пределах максимального времени прибытия
        self.matrix = matrix
        self.ip_val = ip_val

        if not target_weights is None:
            warnings.warn("Учет веса целей в настоящее время не протестирован!", UserWarning)
        self.target_weights = target_weights

        self.state_function = state_function
        self.after_mclp_function = after_mclp_function
        super().__init__(mclp_function, point_selector, stop_case_function, metric_function, names_pattern, start_names_index, **kwargs)

    def __call__(self,
                 env:nx.Graph = None,
                 dynamic_nodes: dict = None,
                 static_nodes: dict = None,
                 area: pd.Series = None,
                 nodes_list:set=None,
                 **kwargs):
        '''
        Запуск работы алгоритма

        ## Аргументы

        `env`:nx.Graph

            Граф улично-дорожной сети

        `dynamic_nodes`: list|dict

            Список стартовых узлов графа в которых размещены
            подразделения оптимальные места которых следует определить.

        `static_nodes`: list|dict = None

            Список стартовых узлов графа в которых размещены
            подразделения изменять размещение которых не следует.

        `area`: pd.Series = None

            Маска узлов графа. Значениями True отмечены узлы графа - цели расчета леса Вороного.
            Если не указана, расчет производится для всех узлов графа.

        `nodes_list`:set=None
        
            Множество узлов графа которые будут рассмотрены в качестве кандидатов.
            Если не указан, то будут рассмотрены все узлы графа.

        ## Возвращает

        `best_dynamic_nodes, best_metric`

            Словарь лучших размещений ({узел: имя}), лучшая метрика.
        '''

        # 0. Проверка корректности пришедших данных
        if not static_nodes is None:
            if not isinstance(static_nodes,dict):
                raise TypeError(f'Аргумент `static_nodes` должен быть типа dict'
                                f'Имеет: {type(static_nodes)}')
        if not dynamic_nodes is None:
            if not isinstance(dynamic_nodes,dict):
                raise TypeError(f'Аргумент `dynamic_nodes` должен быть типа dict'
                                f'Имеет: {type(dynamic_nodes)}')
        if not area is None and not isinstance(area, pd.Series):
            raise TypeError(f'Аргумент `area` должен иметь тип `pd.Series`! Имеет {type(area)}')

        # Если передан список возможных для размещения узлов
        if not nodes_list is None:
            self.matrix = self.matrix[nodes_list]

        # Создание копии матрицы для использования в алгоритме
        matrix_temp = self.matrix.copy()

        # Установка для матрицы признака прикрытой исходя из
        # максимально допустимого времени прибытия
        matrix_temp = matrix_temp <= self.ip_val #- self.delay

        # Если следует учесть веса объектов, умножаем матрицу на них
        if not self.target_weights is None:
            weights = self.target_weights.loc[matrix.index]
            matrix = matrix.mul(weights, axis=0)

        # 0.1. Если статические узлы не указаны - заменяем значение переменной с None на {}
        if static_nodes is None:
            static_nodes = {}
        else:
            # Отброс узлов прикрытых имеющимися подразделениями
            node_column = matrix_temp[static_nodes.keys()]
            matrix_temp = matrix_temp[np.any(node_column, axis=1) == False]
        
        # 0.2. Если динамические узлы не указаны - заменяем значение переменной с None на {}
        if dynamic_nodes is None:
            best_dynamic_nodes = {}
        else:
            best_dynamic_nodes = dynamic_nodes


        # Итерации
        iteration = 0
        name_index = self.start_names_index
        matrix_start_size = matrix_temp.shape[0]
        while len(matrix_temp) > 0:
           
            # ! Проверить !
            # Если статические узлы не были переданы, выполняем расчет
            if static_nodes is None:
                all_nodes = best_dynamic_nodes
            else:
                all_nodes = list_dict_concat(best_dynamic_nodes, static_nodes)
            # if len(all_nodes) == 0:
            #     # 1. Расчет оптимального размещения подразделений
            #     node_id = matrix_temp.columns[np.argmax(np.sum(matrix_temp, axis=0))]
            #     while self.names_pattern.format(name_index) in best_dynamic_nodes.values():
            #         name_index += 1
            #     best_dynamic_nodes[node_id] = self.names_pattern.format(name_index)
            #     node_column = matrix_temp[node_id]
            #     # name_index += 1

            #     # 2. Отброс прикрытых узлов
            #     matrix_temp = matrix_temp[node_column == False]

            # =============== Здесь проверить корректно ли удаляются ===================
            # 1. Расчет оптимального размещения подразделений
            node_id = matrix_temp.columns[np.argmax(np.sum(matrix_temp, axis=0))]
            while self.names_pattern.format(name_index) in best_dynamic_nodes.values():
                    name_index += 1
            best_dynamic_nodes[node_id] = self.names_pattern.format(name_index)
            node_column = matrix_temp[node_id]

            # 2. Отброс прикрытых узлов
            matrix_temp = matrix_temp[node_column == False]
            # print('ИП = ', matrix_start_size, matrix_temp.shape[0], (matrix_start_size - matrix_temp.shape[0])/matrix_start_size)
            # print('ИП = ', (matrix_start_size - matrix_temp.shape[0])/matrix_start_size)
            # ==========================================================================
            
            # 3. Расчет текущей метрики
            # Объединяем статические и динамические узлы
            if static_nodes is None:
                all_nodes = best_dynamic_nodes
            else:
                all_nodes = list_dict_concat(best_dynamic_nodes, static_nodes)
            # Расчет метрики
            current_metric = NodesMetric(self.state_function,
                                         self.metric_function
                                         )(env,
                                           list(all_nodes.keys()),
                                           area=area)
            print('Метрика в теле', current_metric)
            # times, _ = self.state_function(env = env, points = all_nodes)
            # current_metric = self.metric_function(times)
            # 4. Печать отчета расчета
            if not self.after_mclp_function is None:
                self.after_mclp_function(iteration=iteration,
                            best_metric=current_metric,
                            current_metric=current_metric,
                            dynamic_nodes=best_dynamic_nodes,
                            static_nodes=static_nodes)

            # 5. Если достигнута цель расчета, выходим из цикла
            if not self.stop_case_function is None:
                if self.stop_case_function(value=current_metric,
                                iteration=iteration,
                                best_metric=current_metric,
                                current_metric=current_metric,
                                dynamic_nodes=best_dynamic_nodes,
                                static_nodes=static_nodes):
                    return best_dynamic_nodes, current_metric

            # # ========================================================
            # # 1. Расчет оптимального размещения подразделений
            # node_id = matrix_temp.columns[np.argmax(np.sum(matrix_temp, axis=0))]
            # while self.names_pattern.format(name_index) in best_dynamic_nodes.values():
            #         name_index += 1
            # best_dynamic_nodes[node_id] = self.names_pattern.format(name_index)
            # node_column = matrix_temp[node_id]

            # # 2. Отброс прикрытых узлов
            # matrix_temp = matrix_temp[node_column == False]

            # # ========================================================
            iteration += 1
            # name_index += 1

        return best_dynamic_nodes, current_metric

In [72]:
# 3. Сборка и инициализация алгоритма
## Выбор алгоритма моделирования параметров прибытия
state = FirstArrivalUnitState()

## Выбор метрики
metric = CoverIndexBuilding2(buildings_set, ip_val=10) # Используется метрика ИП-10. Для ADD метрика выполняет роль оценки

## Функция остановки
stop_case = lambda value, **kwargs: value >= 100

## Функция выполняемая при добавлении новой ПСЧ
after_mclp = lambda best_metric, dynamic_nodes, **kwargs: print('Подразделений:', len(dynamic_nodes), ' ИП-10:', best_metric)

## Сборка и инициализация алгоритма ADD
add = LSCP_ADD(  
    state_function      = state,
    matrix              = matrix,
    metric_function     = metric,
    # stop_case_function  = stop_case,
    after_mclp_function = after_mclp,
)

# 3. Расчет
best_nodes, best_metric = add(
        env        = G,
    )


# 4. Печать результатов
print('\t', 'ИТОГ:')
print('\t', 'Подразделений:', len(best_nodes), ' ИП-10:', best_metric)

# ## Получение набора данных узлов графа, для сопоставления с полученным лучшим узлом
# g_nodes = ox.graph_to_gdfs(G, edges=False)
# g_nodes.to_file(f'{DATA_PATH}{city_name}.gpkg', layer='add')


# # 5. Отброс мусорных размещений
# best_nodes2, best_metric2 = drop_trash_points(G,
#                 state_function      = state,
#                 metric_function     = metric,
#                 stop_case_function  = stop_case,
#                 dynamic_nodes       = best_nodes,
#                     )
# print('\t', 'После отброса:')
# print('\t', 'Подразделений:', len(best_nodes2), ' ИП-10:', best_metric2)

# # Печать итога
# df = pd.DataFrame({'Подразделений': [len(best_nodes), len(best_nodes2)],
#                     'ИП-10': [best_metric, best_metric2]}, index = ['ADD', 'После отброса'])
# print(tabulate.tabulate(df, headers='keys', tablefmt='grid', showindex=True))



Метрика в теле 90.70501578393547
Подразделений: 1  ИП-10: 90.70501578393547
Метрика в теле 95.61557348298842
Подразделений: 2  ИП-10: 95.61557348298842
Метрика в теле 99.36864258155033
Подразделений: 3  ИП-10: 99.36864258155033
Метрика в теле 99.5089442300947
Подразделений: 4  ИП-10: 99.5089442300947
Метрика в теле 99.61417046650298
Подразделений: 5  ИП-10: 99.61417046650298
Метрика в теле 99.68432129077517
Подразделений: 6  ИП-10: 99.68432129077517
Метрика в теле 99.75447211504735
Подразделений: 7  ИП-10: 99.75447211504735
Метрика в теле 99.78954752718344
Подразделений: 8  ИП-10: 99.78954752718344
Метрика в теле 99.82462293931954
Подразделений: 9  ИП-10: 99.82462293931954
Метрика в теле 99.85969835145563
Подразделений: 10  ИП-10: 99.85969835145563
Метрика в теле 99.89477376359173
Подразделений: 11  ИП-10: 99.89477376359173
Метрика в теле 99.92984917572781
Подразделений: 12  ИП-10: 99.92984917572781
Метрика в теле 99.9649245878639
Подразделений: 13  ИП-10: 99.9649245878639
Метрика в те

In [66]:
times, nearest = state(G, list(best_nodes.keys())[:4])

# print(metric(times))

merged_bld = pd.merge(buildings_set,
                      times,
                      how='left',
                      left_on='node',
                      right_index=True)
merged_bld.shape

sum(merged_bld['times']<=10) / len(buildings_set)

0.9950894423009471

In [ ]:
class CoverIndexBuilding2(MetricBase):
    '''
    !Только для верификации ADD

    Класс-функция расчета индекса прибытия к зданиям
    '''
    def __init__(self,
                 buildings,
                #  f:callable = np.mean,
                 comp_func:callable = max,
                 zero_val=0,
                 ip_val=10,
                 buildings_node_id_field: str = 'node',
                #  buildings_demand_field: str = 'demand',
                 ) -> None:
        '''
        `buildings`: pd.GeoDataFrame
            Геодатафрейм со сведениями о зданиях.
        `f`: callable
            Функция расчета показателя. По-умолчанию = np.mean,
            т.е. вычисляется среднее время следования.
        `comp_func`: callable
            Функция сравнения значений метрики.
            По умолчанию - лучшей считается большая.
        `zero_val`: float = 0
            Значение которое будет возвращено в случае передачи набора данных `route_times` без элементов.
        `buildings_node_id_field`: str = 'node'
            Имя поля в `buildings` содержащего идентификаторы узлов.
        buildings_demand_field: str = 'demand'
            Имя поля в `buildings` содержащего значения спроса
        '''
        if not buildings_node_id_field in buildings.columns:
            raise KeyError(f"Поле `{buildings_node_id_field}` отсутствует в 'buildings'")

        self.buildings = buildings
        # self.f = f
        self.zero_val = zero_val
        self.buildings_node_id_field = buildings_node_id_field
        # self.buildings_demand_field = buildings_demand_field
        self.ip_val = ip_val
        super().__init__(comp_func)

    def __call__(self, state, area=None):
        '''
        `state` (`состояние`): pd.Series
            Серия времен прибытия в разные точки окружения.

        `area`: pd.Series
            Серия данных содержащих маску точек которые должны быть учтены при расчете метрики.
        '''

        if not isinstance(state, (list, pd.Series)):
            raise TypeError(
                f"Аргумент state может быть только типа list или pd.Series"
                f" Имеется {type(state)}"
                )

        
        # Отбор узлов по area (списку узлов которые следует учесть в расчете)
        if not area is None:
            if isinstance(state, pd.Series):
                state_c = state.loc[area]
            else:
                state_c = [x for x in state if x in area]
        else:
            state_c = state

        if len(state_c)==0:
            return self.zero_val

        # Расчет метрики по спросу
        buildings_s = self.buildings.query(f'{self.buildings_node_id_field} in @state_c.keys()') # Вот здесь!!!
        # print(len(self.buildings), len(buildings_s))
        merged_df = pd.merge(buildings_s,
                             state_c,
                             how='left',
                             left_on=self.buildings_node_id_field,
                             right_index=True)
        # Расчет
        state_c = merged_df['times']
        if len(state_c)==0:
            return self.zero_val
        # ip_len = sum([1 for t in state_c if t<=self.ip_val])
        ip_len = sum(state_c <= self.ip_val)
        # tot_len = len(state_c)
        # только для проверки ADD:
        tot_len = len(self.buildings)

        # print(ip_len, tot_len)

        return 100*ip_len/tot_len
    
metric2 = CoverIndexBuilding2(buildings_set, ip_val=10)
print(metric2(times))

99.5089442300947


In [60]:
buildings_set.shape

(500, 11)